# Syria Governorate Dissolve Map by Custom Analytical Region

Overview

Groups the 14 governorates of Syria into four user-defined analytical regions and dissolves the internal governorate boundaries within each region.
This classification was created for this analytical exercise and does not represent an official humanitarian, administrative or operational regional framework.
The resulting regions retain the names, Pcodes and number of their constituent governorates.

シリアの14県を4つの独自分析地域へ分類し、各地域内部の県境をDissolveによって統合します。
この地域分類は本分析のために独自に設定したものであり、公式な人道支援区分、行政区分または活動地域区分を示すものではありません。
Dissolve後の地域には、構成する県の名称、Pcodeおよび県数を保持します。

Objectives

- Assign the 14 governorates to four user-defined analytical regions
- Confirm that every governorate has been classified
- Dissolve the governorate boundaries by analytical region
- Preserve the constituent governorate names, Pcodes and feature counts
- Calculate the area of each region in a projected coordinate system
- Save the derived regions as a GeoPackage
- Display the dissolved regions on an interactive map

- 14県を4つの独自分析地域へ分類する
- すべての県が分類されていることを確認する
- 分析地域ごとに県境をDissolveする
- 構成県名、Pcodeおよび県数を保持する
- 投影座標系を用いて各地域の面積を計算する
- 派生した地域データをGeoPackageとして保存する
- Dissolve後の地域をインタラクティブ地図上に表示する

Workflow

1. Read and validate the national and governorate boundaries
2. Define the custom analytical regions
3. Assign each governorate to an analytical region
4. Confirm the classification and regional membership
5. Aggregate the governorate attributes
6. Dissolve the governorate boundaries by analytical region
7. Confirm the dissolved geometries and attributes
8. Calculate the regional areas in WGS 84 / UTM zone 37N
9. Save and verify the derived GeoPackage
10. Create and export the interactive map

1. 国境データと県境データを読み込み、検証する
2. 独自分析地域を定義する
3. 各県を分析地域へ分類する
4. 分類結果と各地域の構成県を確認する
5. 県名、Pcodeおよび県数を集約する
6. 分析地域ごとに県境をDissolveする
7. Dissolve後のジオメトリと属性を確認する
8. WGS 84 / UTM zone 37Nを用いて地域面積を計算する
9. 派生GeoPackageを保存し、保存結果を確認する
10. インタラクティブ地図を作成して出力する

Data

Administrative boundary data:

- syr_admin0.geojson
- syr_admin1.geojson

Source: HDX OCHA, Syria subnational administrative boundaries

Derived data:

- syr_custom_analytical_regions.gpkg

Technologies

- Python
- GeoPandas
- Folium
- PyProj
- GeoPackage

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import folium
import geopandas as gpd

In [ ]:
# 2
# Define the input and output paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
OUTPUT_DIR = PROJECT_DIR / "outputs"

admin0_path = (
    VECTOR_DIR / "syr_admin0.geojson"
)

admin1_path = (
    VECTOR_DIR / "syr_admin1.geojson"
)

dissolved_regions_path = (
    OUTPUT_DIR / "syr_custom_analytical_regions.gpkg"
)

output_path = (
    PROJECT_DIR / "02_syria_vector_dissolve.html"
)

print(f"Country boundary dataset: {admin0_path}")
print(f"Governorate boundary dataset: {admin1_path}")
print(f"Derived region dataset: {dissolved_regions_path}")
print(f"Interactive map: {output_path}")

In [ ]:
# 3
# Read the national and governorate boundaries
# 国境データと県境データを読み込む

admin0 = gpd.read_file(
    admin0_path
)

admin1 = gpd.read_file(
    admin1_path
)

print(
    "Country features:",
    f"{len(admin0):,}"
)

print(
    "Governorate features:",
    f"{len(admin1):,}"
)

In [ ]:
# 4
# Check the administrative boundary data
# 行政界データの構造とジオメトリを確認する

required_admin0_columns = {
    "adm0_name",
    "adm0_pcode",
    "geometry",
}

required_admin1_columns = {
    "adm1_name",
    "adm1_pcode",
    "geometry",
}

missing_admin0_columns = (
    required_admin0_columns
    - set(admin0.columns)
)

missing_admin1_columns = (
    required_admin1_columns
    - set(admin1.columns)
)

if missing_admin0_columns:
    raise ValueError(
        "The country boundary dataset is missing "
        f"required columns: {sorted(missing_admin0_columns)}"
    )

if missing_admin1_columns:
    raise ValueError(
        "The governorate boundary dataset is missing "
        f"required columns: {sorted(missing_admin1_columns)}"
    )

if len(admin0) != 1:
    raise ValueError(
        "Exactly one country boundary was expected, "
        f"but {len(admin0)} features were found."
    )

if len(admin1) != 14:
    raise ValueError(
        "Exactly 14 governorates were expected, "
        f"but {len(admin1)} features were found."
    )

administrative_datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in (
    administrative_datasets.items()
):

    if dataset.crs is None:
        raise ValueError(
            f"{dataset_name} has no defined CRS."
        )

    if dataset.geometry.isna().any():
        raise ValueError(
            f"{dataset_name} contains missing geometries."
        )

    if dataset.geometry.is_empty.any():
        raise ValueError(
            f"{dataset_name} contains empty geometries."
        )

    if not dataset.geometry.is_valid.all():
        invalid_count = int(
            (~dataset.geometry.is_valid).sum()
        )

        raise ValueError(
            f"{dataset_name} contains "
            f"{invalid_count:,} invalid geometries."
        )

if admin0.crs != admin1.crs:
    raise ValueError(
        "The country and governorate boundaries "
        "must use the same CRS."
    )

if admin1.crs.to_epsg() != 4326:
    raise ValueError(
        "The administrative boundaries were expected "
        f"to use EPSG:4326, but their CRS is {admin1.crs}."
    )

allowed_geometry_types = {
    "Polygon",
    "MultiPolygon",
}

for dataset_name, dataset in (
    administrative_datasets.items()
):

    unexpected_geometry_types = (
        set(dataset.geom_type.unique())
        - allowed_geometry_types
    )

    if unexpected_geometry_types:
        raise ValueError(
            f"{dataset_name} contains unexpected "
            "geometry types: "
            f"{sorted(unexpected_geometry_types)}"
        )

print(
    "Administrative boundary validation: passed"
)

print(
    f"Country boundary CRS: {admin0.crs}"
)

print(
    f"Governorate boundary CRS: {admin1.crs}"
)

print(
    "Governorate geometry types:"
)

print(
    admin1.geom_type.value_counts()
)

In [ ]:
# 5
# Define the custom analytical regions
# 独自分析地域の分類を定義する

ANALYTICAL_REGION_MAP = {
    "Aleppo": "Northwest",
    "Idleb": "Northwest",
    "Lattakia": "Northwest",

    "Al-Hasakeh": "Northeast",
    "Ar-Raqqa": "Northeast",
    "Deir-ez-Zor": "Northeast",

    "Hama": "Central",
    "Homs": "Central",
    "Tartous": "Central",

    "As-Sweida": "South",
    "Damascus": "South",
    "Dar'a": "South",
    "Quneitra": "South",
    "Rural Damascus": "South",
}

EXPECTED_ANALYTICAL_REGIONS = {
    "Northwest",
    "Northeast",
    "Central",
    "South",
}

print(
    "Defined governorate classifications:",
    f"{len(ANALYTICAL_REGION_MAP):,}"
)

print(
    "Defined analytical regions:",
    sorted(EXPECTED_ANALYTICAL_REGIONS)
)

In [ ]:
# 6
# Assign each governorate to an analytical region
# 各県を分析地域へ分類する

admin1_classified = (
    admin1.copy()
)

admin1_classified[
    "analytical_region"
] = (
    admin1_classified[
        "adm1_name"
    ].map(
        ANALYTICAL_REGION_MAP
    )
)

print(
    admin1_classified[
        [
            "adm1_name",
            "adm1_pcode",
            "analytical_region",
        ]
    ].sort_values(
        [
            "analytical_region",
            "adm1_name",
        ]
    )
)

In [ ]:
# 7
# Check that all governorates are classified
# すべての県が分類されていることを確認する

governorate_names = set(
    admin1_classified[
        "adm1_name"
    ]
)

classification_names = set(
    ANALYTICAL_REGION_MAP
)

unclassified_governorates = (
    governorate_names
    - classification_names
)

unknown_classification_names = (
    classification_names
    - governorate_names
)

if unclassified_governorates:
    raise ValueError(
        "The following governorates have no "
        "analytical region classification: "
        f"{sorted(unclassified_governorates)}"
    )

if unknown_classification_names:
    raise ValueError(
        "The classification contains governorate names "
        "that are not present in the boundary data: "
        f"{sorted(unknown_classification_names)}"
    )

duplicated_governorate_names = sorted(
    admin1_classified.loc[
        admin1_classified[
            "adm1_name"
        ].duplicated(
            keep=False
        ),
        "adm1_name",
    ].unique()
)

if duplicated_governorate_names:
    raise ValueError(
        "Duplicate governorate names were found: "
        f"{duplicated_governorate_names}"
    )

actual_analytical_regions = set(
    admin1_classified[
        "analytical_region"
    ].unique()
)

if (
    actual_analytical_regions
    != EXPECTED_ANALYTICAL_REGIONS
):
    raise ValueError(
        "The assigned analytical regions do not match "
        "the defined regions. "
        f"Assigned: {sorted(actual_analytical_regions)}; "
        f"Defined: {sorted(EXPECTED_ANALYTICAL_REGIONS)}"
    )

region_membership_summary = (
    admin1_classified.groupby(
        "analytical_region"
    )[
        "adm1_name"
    ].agg(
        governorate_count="count",
        governorate_names=lambda names: ", ".join(
            sorted(names)
        ),
    )
)

print(
    "Governorate classification: passed"
)

print(
    region_membership_summary
)

In [ ]:
# 8
# Aggregate the governorate attributes by region
# 分析地域ごとに県の属性を集約する

region_attribute_summary = (
    admin1_classified.groupby(
        "analytical_region",
        as_index=False,
    ).agg(
        governorate_count=(
            "adm1_name",
            "count",
        ),
        governorate_names=(
            "adm1_name",
            lambda names: ", ".join(
                sorted(names)
            ),
        ),
        governorate_pcodes=(
            "adm1_pcode",
            lambda pcodes: ", ".join(
                sorted(pcodes)
            ),
        ),
    )
)

print(
    region_attribute_summary
)

In [ ]:
# 9
# Dissolve the governorate boundaries by region
# 分析地域ごとに県境を統合する

dissolved_regions = (
    admin1_classified[
        [
            "analytical_region",
            "geometry",
        ]
    ].dissolve(
        by="analytical_region",
        as_index=False,
    )
)

print(
    "Dissolved analytical regions:",
    f"{len(dissolved_regions):,}"
)

print(
    dissolved_regions.geom_type.value_counts()
)

In [ ]:
# 10
# Add the aggregated attributes to the dissolved regions
# Dissolve後の地域へ集約した属性を追加する

dissolved_regions = (
    dissolved_regions.merge(
        region_attribute_summary,
        on="analytical_region",
        how="left",
        validate="one_to_one",
    )
)

region_display_order = {
    "Northwest": 1,
    "Northeast": 2,
    "Central": 3,
    "South": 4,
}

dissolved_regions[
    "_display_order"
] = (
    dissolved_regions[
        "analytical_region"
    ].map(
        region_display_order
    )
)

dissolved_regions = (
    dissolved_regions.sort_values(
        "_display_order"
    ).drop(
        columns="_display_order"
    ).reset_index(
        drop=True
    )
)

print(
    dissolved_regions[
        [
            "analytical_region",
            "governorate_count",
            "governorate_names",
            "governorate_pcodes",
        ]
    ]
)

In [ ]:
# 11
# Check the dissolved regions
# Dissolve後の地域と集約属性を確認する

required_dissolved_attributes = [
    "governorate_count",
    "governorate_names",
    "governorate_pcodes",
]

if len(dissolved_regions) != 4:
    raise ValueError(
        "Exactly four dissolved regions were expected, "
        f"but {len(dissolved_regions)} were created."
    )

if dissolved_regions.crs != admin1_classified.crs:
    raise ValueError(
        "The dissolved regions do not retain "
        "the source governorate CRS."
    )

if dissolved_regions[
    required_dissolved_attributes
].isna().any().any():
    raise ValueError(
        "One or more dissolved regions are missing "
        "aggregated governorate attributes."
    )

aggregated_governorate_count = int(
    dissolved_regions[
        "governorate_count"
    ].sum()
)

if aggregated_governorate_count != len(
    admin1_classified
):
    raise ValueError(
        "The aggregated governorate count does not "
        "equal the source governorate count."
    )

if dissolved_regions.geometry.isna().any():
    raise ValueError(
        "The dissolved region data contains "
        "missing geometries."
    )

if dissolved_regions.geometry.is_empty.any():
    raise ValueError(
        "The dissolved region data contains "
        "empty geometries."
    )

if not dissolved_regions.geometry.is_valid.all():
    invalid_region_count = int(
        (
            ~dissolved_regions.geometry.is_valid
        ).sum()
    )

    raise ValueError(
        "The dissolved region data contains "
        f"{invalid_region_count:,} invalid geometries."
    )

source_governorate_union = (
    admin1_classified.geometry.union_all()
)

dissolved_region_union = (
    dissolved_regions.geometry.union_all()
)

if not dissolved_region_union.equals(
    source_governorate_union
):
    raise ValueError(
        "The dissolved regions do not preserve "
        "the source governorate coverage."
    )

print(
    "Dissolved region validation: passed"
)

print(
    "Dissolved regions:",
    f"{len(dissolved_regions):,}"
)

print(
    "Governorates represented:",
    f"{aggregated_governorate_count:,}"
)

print(
    "Dissolved region CRS:",
    dissolved_regions.crs
)

In [ ]:
# 12
# Calculate the area of each analytical region
# 各分析地域の面積を平方キロメートルで計算する

AREA_CRS = "EPSG:32637"

dissolved_regions_projected = (
    dissolved_regions.to_crs(
        AREA_CRS
    )
)

if not dissolved_regions_projected.crs.is_projected:
    raise ValueError(
        "The area calculation requires "
        "a projected coordinate system."
    )

region_area_table = (
    dissolved_regions_projected[
        [
            "analytical_region",
        ]
    ].copy()
)

region_area_table[
    "area_sqkm"
] = (
    dissolved_regions_projected.geometry.area
    / 1_000_000
)

dissolved_regions = (
    dissolved_regions.merge(
        region_area_table,
        on="analytical_region",
        how="left",
        validate="one_to_one",
    )
)

if dissolved_regions[
    "area_sqkm"
].isna().any():
    raise ValueError(
        "One or more analytical regions have "
        "a missing area value."
    )

if (
    dissolved_regions[
        "area_sqkm"
    ] <= 0
).any():
    raise ValueError(
        "All analytical regions must have "
        "a positive area."
    )

print(
    dissolved_regions[
        [
            "analytical_region",
            "governorate_count",
            "area_sqkm",
        ]
    ].to_string(
        index=False,
        formatters={
            "area_sqkm": lambda value: (
                f"{value:,.2f}"
            )
        },
    )
)

In [ ]:
# 13
# Save the dissolved regions as a GeoPackage
# Dissolve後の地域をGeoPackageとして保存する

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dissolved_regions.to_file(
    dissolved_regions_path,
    layer="custom_analytical_regions",
    driver="GPKG",
    index=False,
)

print(
    "Derived analytical region dataset saved to:",
    dissolved_regions_path,
)

In [ ]:
# 14
# Read and check the saved GeoPackage
# 保存したGeoPackageを読み込み、内容を確認する

saved_dissolved_regions = gpd.read_file(
    dissolved_regions_path,
    layer="custom_analytical_regions",
)

required_saved_attributes = (
    required_dissolved_attributes
    + [
        "area_sqkm",
    ]
)

if len(saved_dissolved_regions) != 4:
    raise ValueError(
        "The saved GeoPackage does not contain "
        "exactly four analytical regions."
    )

if (
    saved_dissolved_regions.crs
    != dissolved_regions.crs
):
    raise ValueError(
        "The saved CRS does not match "
        "the in-memory result."
    )

if saved_dissolved_regions[
    required_saved_attributes
].isna().any().any():
    raise ValueError(
        "The saved GeoPackage contains "
        "missing analytical region attributes."
    )

if saved_dissolved_regions.geometry.isna().any():
    raise ValueError(
        "The saved GeoPackage contains "
        "missing geometries."
    )

if saved_dissolved_regions.geometry.is_empty.any():
    raise ValueError(
        "The saved GeoPackage contains "
        "empty geometries."
    )

if not saved_dissolved_regions.geometry.is_valid.all():
    invalid_saved_count = int(
        (
            ~saved_dissolved_regions.geometry.is_valid
        ).sum()
    )

    raise ValueError(
        "The saved GeoPackage contains "
        f"{invalid_saved_count:,} invalid geometries."
    )

print(
    "Saved GeoPackage validation: passed"
)

print(
    "Saved analytical regions:",
    f"{len(saved_dissolved_regions):,}"
)

print(
    "Saved CRS:",
    saved_dissolved_regions.crs
)

print(
    saved_dissolved_regions[
        [
            "analytical_region",
            "governorate_count",
            "governorate_names",
            "governorate_pcodes",
            "area_sqkm",
        ]
    ]
)

In [ ]:
# 15
# Define the region colours and create the basemap
# 地域別の色を設定し、ベースマップを作成する

REGION_COLORS = {
    "Northwest": "#1b395c",
    "Northeast": "#8b705f",
    "Central": "#e9c46a",
    "South": "#27606c",
}

if set(
    REGION_COLORS
) != EXPECTED_ANALYTICAL_REGIONS:
    raise ValueError(
        "The region colour definitions do not match "
        "the defined analytical regions."
    )

m = folium.Map(
    location=[
        34.8,
        38.5,
    ],
    zoom_start=7,
    tiles=None,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">'
        "CARTO</a>"
    ),
    name="CARTO Light — No Labels",
    overlay=False,
    control=True,
).add_to(
    m
)

min_x, min_y, max_x, max_y = (
    admin0.total_bounds
)

syria_view_bounds = [
    [
        min_y,
        min_x,
    ],
    [
        max_y,
        max_x,
    ],
]

m.fit_bounds(
    syria_view_bounds,
    padding=(35, 35),
    max_zoom=7,
)

In [ ]:
# 16
# Add the dissolved analytical regions
# Dissolve後の分析地域を地図へ追加する

regions_map = (
    saved_dissolved_regions.copy()
)

regions_map[
    "area_sqkm_display"
] = (
    regions_map[
        "area_sqkm"
    ].map(
        lambda value: f"{value:,.2f} km²"
    )
)

folium.GeoJson(
    data=regions_map,
    name="Custom Analytical Regions",
    show=True,
    style_function=lambda feature: {
        "fillColor": REGION_COLORS[
            feature[
                "properties"
            ][
                "analytical_region"
            ]
        ],
        "color": "#444444",
        "weight": 2,
        "fillOpacity": 0.48,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 3,
        "fillOpacity": 0.65,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "analytical_region",
            "governorate_count",
            "governorate_names",
            "governorate_pcodes",
            "area_sqkm_display",
        ],
        aliases=[
            "Analytical region:",
            "Governorate count:",
            "Governorates:",
            "Governorate Pcodes:",
            "Calculated area:",
        ],
        localize=True,
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

In [ ]:
# 17
# Add the original administrative boundaries
# 比較用に元の国境と県境を追加する

folium.GeoJson(
    data=admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ],
    name="Original Governorate Boundaries",
    show=False,
    style_function=lambda feature: {
        "color": "#666666",
        "weight": 1,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

folium.GeoJson(
    data=admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ],
    name="Syria Boundary",
    show=True,
    style_function=lambda feature: {
        "color": "#222222",
        "weight": 2.5,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

In [ ]:
# 18
# Add the neighbouring country labels
# 周辺国名を地図へ追加する

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKIYE": [
        37.5,
        37.5,
    ],
    "IRAQ": [
        34.5,
        42.0,
    ],
    "JORDAN": [
        31.9,
        36.5,
    ],
    "LEBANON": [
        34.2,
        35.0,
    ],
}

for country_name, coordinates in (
    neighbour_labels.items()
):

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 120px;
                margin-left: -60px;
                color: #666666;
                font-size: 14pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                text-shadow:
                    -1px -1px 0 white,
                    1px -1px 0 white,
                    -1px 1px 0 white,
                    1px 1px 0 white;
            ">
                {country_name}
            </div>
            """
        ),
    ).add_to(
        neighbour_label_layer
    )

neighbour_label_layer.add_to(
    m
)

In [ ]:
# 19
# Add a label inside each analytical region
# 各分析地域の内側に地域名を表示する

region_label_layer = folium.FeatureGroup(
    name="Analytical Region Labels",
    show=True,
)

for _, region in regions_map.iterrows():

    label_point = (
        region.geometry.representative_point()
    )

    region_name = (
        region[
            "analytical_region"
        ]
    )

    region_color = (
        REGION_COLORS[
            region_name
        ]
    )

    folium.Marker(
        location=[
            label_point.y,
            label_point.x,
        ],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 140px;
                margin-left: -70px;
                padding: 3px 5px;
                color: #222222;
                background-color: rgba(255, 255, 255, 0.76);
                border: 2px solid {region_color};
                border-radius: 5px;
                font-size: 12pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                box-shadow: 0 0 4px rgba(0, 0, 0, 0.25);
            ">
                {region_name}
            </div>
            """
        ),
    ).add_to(
        region_label_layer
    )

region_label_layer.add_to(
    m
)

In [ ]:
# 20
# Add the map information and source panel
# 地図の説明、分析上の注意および出典を追加する

total_calculated_area = float(
    saved_dissolved_regions[
        "area_sqkm"
    ].sum()
)

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 430px;
    min-height: 225px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #6a4c93;
        font-weight: bold;
    ">
        Governorate Dissolve by Custom Analytical Region
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        The 14 governorates are grouped into four
        user-defined analytical regions.
        Internal governorate boundaries are removed
        through the dissolve operation.
        This classification was created for this
        analytical exercise and does not represent
        an official humanitarian, administrative
        or operational regional framework.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Boundary source:
        <b>HDX OCHA</b><br>

        Source governorates:
        <b>{len(admin1):,}</b><br>

        Analytical regions:
        <b>{len(saved_dissolved_regions):,}</b><br>

        Calculated total area:
        <b>{total_calculated_area:,.2f} km²</b><br>

        Method:
        Attribute Classification / Dissolve / Area Calculation
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        information_panel_html
    )
)

In [ ]:
# 21
# Add the analytical region colour legend
# 分析地域の色を示す凡例を追加する

legend_items_html = ""

for region_name in [
    "Northwest",
    "Northeast",
    "Central",
    "South",
]:

    region_color = (
        REGION_COLORS[
            region_name
        ]
    )

    legend_items_html += f"""
    <div style="
        display: flex;
        align-items: center;
        margin-top: 8px;
    ">
        <span style="
            display: inline-block;
            width: 28px;
            height: 15px;
            margin-right: 9px;
            background-color: {region_color};
            border: 1px solid #444444;
            opacity: 0.75;
        "></span>

        {region_name}
    </div>
    """

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    right: 40px;
    width: 245px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    font-size: 12px;
    z-index: 9999;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.2);
">
    <b style="font-size: 13px;">
        Custom Analytical Regions
    </b>

    {legend_items_html}

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        color: #555555;
        border-top: 1px solid #aaaaaa;
        line-height: 1.35;
    ">
        User-defined analytical classification<br>
        Not an official regional framework
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        legend_html
    )
)

In [ ]:
# 22
# Add the layer control
# 地図レイヤーの表示と非表示を切り替える機能を追加する

folium.LayerControl(
    collapsed=False,
).add_to(
    m
)

In [ ]:
# 23
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m